# MLflow 실험 추적

MLflow를 사용하여 ML 실험을 추적하고 관리합니다.

## 학습 목표
1. MLflow 기본 사용법
2. 실험 추적 및 로깅
3. 모델 레지스트리
4. 모델 배포

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.datasets import load_iris
import pandas as pd
import numpy as np

print(f"MLflow version: {mlflow.__version__}")

## 1. MLflow 설정

In [ ]:
# MLflow 추적 서버 설정 (로컬)
mlflow.set_tracking_uri("sqlite:///mlflow.db")

# 실험 생성
experiment_name = "iris_classification"
mlflow.set_experiment(experiment_name)

print(f"Experiment: {experiment_name}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

## 2. 데이터 준비

In [ ]:
# Iris 데이터셋 로드
iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 3. MLflow로 실험 추적

In [ ]:
def train_and_log(n_estimators, max_depth, min_samples_split):
    with mlflow.start_run():
        # 하이퍼파라미터 로깅
        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)
        mlflow.log_param("min_samples_split", min_samples_split)
        
        # 모델 학습
        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            random_state=42
        )
        model.fit(X_train, y_train)
        
        # 예측
        y_pred = model.predict(X_test)
        
        # 메트릭 계산 및 로깅
        accuracy = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='weighted')
        
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("f1_score", f1)
        
        # 모델 저장
        mlflow.sklearn.log_model(model, "model")
        
        print(f"n_estimators={n_estimators}, max_depth={max_depth}, "
              f"min_samples_split={min_samples_split}")
        print(f"Accuracy: {accuracy:.4f}, F1: {f1:.4f}")
        
        return accuracy, f1

# 여러 설정으로 실험
experiments = [
    (50, 5, 2),
    (100, 10, 2),
    (100, None, 5),
    (200, 15, 2),
]

for n_est, depth, min_split in experiments:
    train_and_log(n_est, depth, min_split)
    print()

## 4. 실험 결과 조회

In [ ]:
# 실험 결과 조회
experiment = mlflow.get_experiment_by_name(experiment_name)
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

# 결과 표시
results = runs[['run_id', 'params.n_estimators', 'params.max_depth', 
                'metrics.accuracy', 'metrics.f1_score']]
print(results.to_string(index=False))

In [ ]:
# 최고 성능 모델 찾기
best_run = runs.loc[runs['metrics.accuracy'].idxmax()]
print(f"\n최고 성능 모델:")
print(f"Run ID: {best_run['run_id']}")
print(f"Accuracy: {best_run['metrics.accuracy']:.4f}")

## 5. 모델 로드 및 예측

In [ ]:
# 최고 성능 모델 로드
best_run_id = best_run['run_id']
model_uri = f"runs:/{best_run_id}/model"

loaded_model = mlflow.sklearn.load_model(model_uri)

# 예측
sample = X_test.iloc[:5]
predictions = loaded_model.predict(sample)

print("샘플 예측:")
for i, (pred, actual) in enumerate(zip(predictions, y_test[:5])):
    print(f"  {i+1}. 예측: {iris.target_names[pred]}, 실제: {iris.target_names[actual]}")

## 6. 아티팩트 로깅

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import seaborn as sns

with mlflow.start_run():
    # 모델 학습
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    # 혼동 행렬 시각화
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=iris.target_names,
                yticklabels=iris.target_names)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion Matrix')
    plt.savefig('confusion_matrix.png')
    
    # 아티팩트 로깅
    mlflow.log_artifact('confusion_matrix.png')
    
    # 특성 중요도 저장
    importance_df = pd.DataFrame({
        'feature': iris.feature_names,
        'importance': model.feature_importances_
    }).sort_values('importance', ascending=False)
    importance_df.to_csv('feature_importance.csv', index=False)
    mlflow.log_artifact('feature_importance.csv')
    
    print("아티팩트 로깅 완료!")

## 연습 문제

1. 다른 분류기 (XGBoost, LightGBM)를 추가하고 비교해보세요.
2. MLflow UI를 실행하여 결과를 시각화해보세요 (`mlflow ui`).
3. 모델을 레지스트리에 등록해보세요.
4. 하이퍼파라미터 튜닝을 자동화해보세요 (Optuna + MLflow).